# Leakage experiment (restart-resilient)

Trains YOLO11n on a randomly partitioned version of the same corpus and
compares against the spatially disjoint baseline. Checkpoints write directly
to Drive, so a recycled runtime costs at most five epochs.

Run cells in order. If the runtime is recycled mid-training, re-run cells
1--4 and then the resume cell at the bottom rather than the training cell.

In [ ]:
# 1. Setup
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/cotton_bench'
!pip install -q ultralytics
import os, json, glob, random, shutil
import numpy as np
print('ok')

In [ ]:
# 2. Unpack tiles and silver labels (~10 min)
for c in ['pre','post']:
    n = len(glob.glob(f'/content/imgs_{c}/train_tiles/*.jpg'))
    if n == 0:
        !rm -rf /content/imgs_{c}
        !unzip -q {DRIVE}/cotton_train_{c}.zip -d /content/imgs_{c}
    n = len(glob.glob(f'/content/lbls_{c}/silver_{c}_labels/*.txt'))
    if n == 0:
        !rm -rf /content/lbls_{c}
        !unzip -q {DRIVE}/silver_{c}/labels.zip -d /content/lbls_{c}

for c in ['pre','post']:
    print(c,
          len(glob.glob(f'/content/imgs_{c}/train_tiles/*.jpg')), 'images',
          len(glob.glob(f'/content/lbls_{c}/silver_{c}_labels/*.txt')), 'labels')

In [ ]:
# 3. Random partition (seeded, reproducible)
pool = {}
for c in ['pre','post']:
    p = json.load(open(f'{DRIVE}/silver_{c}/raw_preds.json'))
    for t, boxes in p.items():
        pool[t] = (c, len(boxes))
print('candidate tiles:', len(pool))

random.seed(101)
all_tiles = sorted(pool)
random.shuffle(all_tiles)

N_TRAIN, N_TEST, N_VAL = 30000, 1215, 1503
rand_test  = all_tiles[:N_TEST]
rand_train = all_tiles[N_TEST:N_TEST+N_TRAIN]
rand_val   = all_tiles[N_TEST+N_TRAIN:N_TEST+N_TRAIN+N_VAL]
print(f'train {len(rand_train)}, val {len(rand_val)}, test {len(rand_test)}')
json.dump(dict(train=rand_train, val=rand_val, test=rand_test),
          open(f'{DRIVE}/random_split.json','w'))

In [ ]:
# 4. Assemble dataset
!rm -rf /content/ds_rand
splits = dict(train=rand_train, val=rand_val)
for s in splits:
    for sub in ['images','labels']:
        os.makedirs(f'/content/ds_rand/{s}/{sub}', exist_ok=True)

missing = 0
for s, tiles in splits.items():
    for t in tiles:
        c = pool[t][0]
        si = f'/content/imgs_{c}/train_tiles/{t}'
        sl = f'/content/lbls_{c}/silver_{c}_labels/{os.path.splitext(t)[0]}.txt'
        if not (os.path.exists(si) and os.path.exists(sl)):
            missing += 1; continue
        os.symlink(si, f'/content/ds_rand/{s}/images/{t}')
        shutil.copy(sl, f'/content/ds_rand/{s}/labels/')

with open('/content/ds_rand/data.yaml','w') as f:
    f.write('path: /content/ds_rand\ntrain: train/images\nval: val/images\n'
            'names:\n  0: cotton_boll\n')
print('train', len(os.listdir('/content/ds_rand/train/images')),
      'val', len(os.listdir('/content/ds_rand/val/images')), 'missing', missing)

## 5. Train

30 epochs rather than 100. The spatial baselines gained under $0.01$
mAP50-95 between epochs 30 and 100, and this control measures a difference
between protocols rather than an absolute score. Checkpoints go to Drive.

In [ ]:
MODEL, RUN = 'yolo11n.pt', 'yolo11n_randomsplit'
from ultralytics import YOLO

m = YOLO(MODEL)
m.train(data='/content/ds_rand/data.yaml', epochs=30, imgsz=640, batch=16,
        patience=20, project=f'{DRIVE}/runs', name=RUN,
        max_det=1000, seed=42, val=True, plots=True, save_period=5,
        exist_ok=True)

dst = f'{DRIVE}/baselines/{RUN}'
os.makedirs(dst, exist_ok=True)
shutil.copy(f'{DRIVE}/runs/{RUN}/weights/best.pt', f'{dst}/best.pt')
for f in ['results.csv','args.yaml']:
    p = f'{DRIVE}/runs/{RUN}/{f}'
    if os.path.exists(p): shutil.copy(p, dst)
print('saved to', dst)

### Resume cell

Use this instead of the training cell if the runtime was recycled mid-run.
Re-run cells 1--4 first.

In [ ]:
from ultralytics import YOLO
YOLO(f'{DRIVE}/runs/yolo11n_randomsplit/weights/last.pt').train(resume=True)

## 6. Evaluate

The random-split model is scored on its own held-out tiles against silver
labels. The spatial baseline is scored on training-region tiles it never saw,
also against silver labels, so the two protocols are compared on equivalent
ground truth.

In [ ]:
from tqdm import tqdm

def load_silver(tile):
    c = pool[tile][0]
    p = f'/content/lbls_{c}/silver_{c}_labels/{os.path.splitext(tile)[0]}.txt'
    out = []
    if os.path.exists(p):
        for line in open(p):
            q = line.split()
            if len(q) == 5:
                _, cx, cy, w, h = map(float, q)
                out.append([(cx-w/2)*640, (cy-h/2)*640,
                            (cx+w/2)*640, (cy+h/2)*640])
    return out

def path_of(t): return f'/content/imgs_{pool[t][0]}/train_tiles/{t}'

def predict(weights, tiles_list):
    mdl = YOLO(weights)
    out = {}
    for i in tqdm(range(0, len(tiles_list), 64)):
        chunk = tiles_list[i:i+64]
        res = mdl.predict([path_of(t) for t in chunk], imgsz=640, conf=0.001,
                          iou=0.7, max_det=1000, verbose=False)
        for t, r in zip(chunk, res):
            b = r.boxes
            out[t] = [[float(x1),float(y1),float(x2),float(y2),float(c)]
                      for (x1,y1,x2,y2), c in zip(b.xyxy.cpu().numpy(),
                                                  b.conf.cpu().numpy())]
    return out

def iou_matrix(A, B):
    if len(A) == 0 or len(B) == 0: return np.zeros((len(A), len(B)), np.float32)
    A = np.asarray(A, np.float32); B = np.asarray(B, np.float32)
    ix1 = np.maximum(A[:,0,None], B[None,:,0]); iy1 = np.maximum(A[:,1,None], B[None,:,1])
    ix2 = np.minimum(A[:,2,None], B[None,:,2]); iy2 = np.minimum(A[:,3,None], B[None,:,3])
    inter = np.clip(ix2-ix1,0,None)*np.clip(iy2-iy1,0,None)
    aA = ((A[:,2]-A[:,0])*(A[:,3]-A[:,1]))[:,None]
    aB = ((B[:,2]-B[:,0])*(B[:,3]-B[:,1]))[None,:]
    u = aA + aB - inter
    return np.divide(inter, u, out=np.zeros_like(inter), where=u>0)

def prf(preds, gtd, thr, keys, iou_t=0.5):
    TP = FP = FN = 0
    for t in keys:
        g = gtd.get(t, [])
        p = sorted([b for b in preds.get(t, []) if b[4] >= thr], key=lambda b: -b[4])
        if not p: FN += len(g); continue
        if not g: FP += len(p); continue
        M = iou_matrix([b[:4] for b in p], g)
        taken = set(); tp = 0
        for i in range(len(p)):
            for cand in np.argsort(-M[i]):
                if M[i, cand] < iou_t: break
                if cand not in taken:
                    taken.add(int(cand)); tp += 1; break
        TP += tp; FP += len(p)-tp; FN += len(g)-tp
    P = TP/max(TP+FP,1); R = TP/max(TP+FN,1)
    return dict(P=round(P,4), R=round(R,4), F1=round(2*P*R/max(P+R,1e-9),4))

In [ ]:
RUN = 'yolo11n_randomsplit'

import os
src = f'{DRIVE}/runs/{RUN}/weights/best.pt'
dst_dir = f'{DRIVE}/baselines/{RUN}'
print('training output exists:', os.path.exists(src))
print('baselines copy exists:', os.path.exists(f'{dst_dir}/best.pt'))

In [ ]:
import shutil, os
RUN = 'yolo11n_randomsplit'
dst_dir = f'{DRIVE}/baselines/{RUN}'
os.makedirs(dst_dir, exist_ok=True)
shutil.copy(f'{DRIVE}/runs/{RUN}/weights/best.pt', f'{dst_dir}/best.pt')
for f in ['results.csv', 'args.yaml']:
    p = f'{DRIVE}/runs/{RUN}/{f}'
    if os.path.exists(p): shutil.copy(p, dst_dir)
print('copied:', os.listdir(dst_dir))

In [ ]:
import pandas as pd

# Random-split model on its own test tiles
gt_rand = {t: load_silver(t) for t in rand_test}
pred_rand = predict(f'{DRIVE}/baselines/{RUN}/best.pt', rand_test)

# Spatial model on training-region tiles it never saw
subset = json.load(open(f'{DRIVE}/baseline_subset_30k.json'))
used = set(subset['train']) | set(subset['val'])
spatial_holdout = [t for t in all_tiles if t not in used][:1215]
gt_sp = {t: load_silver(t) for t in spatial_holdout}
pred_sp = predict(f'{DRIVE}/baselines/yolo11n/best.pt', spatial_holdout)

rows = []
for thr in [0.25, 0.30, 0.35]:
    rows.append(dict(protocol='random split', conf=thr,
                     **prf(pred_rand, gt_rand, thr, rand_test)))
    rows.append(dict(protocol='spatial split', conf=thr,
                     **prf(pred_sp, gt_sp, thr, spatial_holdout)))
res = pd.DataFrame(rows).sort_values(['conf','protocol'])
print(res.to_string(index=False))
res.to_csv(f'{DRIVE}/results_leakage.csv', index=False)

The difference between the two rows at a matched threshold is the score
inflation attributable to partitioning by file rather than by ground
position. Both models were trained on 30,000 tiles under otherwise identical
recipes; only the assignment of held-out data differs.